# Plot FIM Itemset Comparison

Loads `data/fim_compare_runs_latest.csv` and plots mean runtime with std error bars for itemset-only benchmarking.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

current_dir = Path.cwd()
if not (current_dir / 'data').exists():
    candidate = (Path.cwd() / 'notebooks' / 'profiling' / 'fim_compare').resolve()
    if (candidate / 'data').exists():
        current_dir = candidate

csv_path = current_dir / 'data' / 'fim_compare_runs_latest.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'Missing file: {csv_path}. Run run_fim_comparison.ipynb first.')

df = pd.read_csv(csv_path)
print('Loaded:', csv_path)
print('Rows:', len(df))
df.head()


In [ ]:
ok = df[df['status'] == 'ok'].copy()
if ok.empty:
    raise RuntimeError('No successful runs (status=ok) in the loaded file.')

summary = (
    ok.groupby(['dataset_path', 'algorithm'], as_index=False)
      .agg(
          runs=('elapsed_seconds', 'count'),
          mean_s=('elapsed_seconds', 'mean'),
          median_s=('elapsed_seconds', 'median'),
          std_s=('elapsed_seconds', lambda x: float(x.std(ddof=1)) if len(x) > 1 else 0.0),
      )
      .sort_values(['dataset_path', 'algorithm'])
)
summary


In [ ]:
for dataset_path, group in summary.groupby('dataset_path'):
    group = group.sort_values('mean_s').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(group['algorithm'], group['mean_s'], yerr=group['std_s'], capsize=5)
    ax.set_title(f'Frequent itemset runtime comparison ({Path(dataset_path).name})')
    ax.set_ylabel('Mean elapsed seconds')
    ax.set_xlabel('Algorithm')
    ax.grid(axis='y', alpha=0.3)

    for bar, mean_val in zip(bars, group['mean_s']):
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height(),
            f'{mean_val:.3f}s',
            ha='center',
            va='bottom',
            fontsize=10,
        )

    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()


In [ ]:
errors = df[df['status'] != 'ok'][['dataset_path', 'algorithm', 'run_index', 'status', 'note']]
errors
